In [1]:
# =============================================================================
# HUMOB / SIGSPATIAL Cup 2025
# Ensemble of Predictive Pattern-Matching (PPM) Models
#
# FINAL FIXED IMPLEMENTATION
#
# Strategies:
# 1. strategy_1 (PPM): Baseline "exact match" counting.
# 2. strategy_2 (cosine_time): FIXED. Cosine similarity on *matching time slots*.
# 3. strategy_3 (cosine_poi_time): FIXED. Cosine similarity on *matching time
#    slots*, with a *weight boost* for POI matches.
# =============================================================================

# ----------------------------
# Requirements (run once)
# ----------------------------
print("Installing/checking requirements...")
%pip install -q git+https://github.com/yahoojapan/geobleu.git tqdm scikit-learn pandas

# ----------------------------
# Imports
# ----------------------------
import os
import shutil
import gc
import json
import time
import pandas as pd
import numpy as np
from collections import defaultdict, Counter
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
import warnings

# Suppress harmless warnings (e.g., from empty cosine sims)
warnings.filterwarnings('ignore', category=RuntimeWarning) 

# geobleu
try:
    from geobleu import calc_geobleu_single, calc_geobleu_bulk
    print("GeoBLEU imported successfully.")
except Exception as e:
    print(f"Failed to import geobleu: {e}")
    calc_geobleu_bulk = None

# ----------------------------
# Global configs
# ----------------------------
DATA_DIR = "/kaggle/input/humob-data/15313913"
CITIES = ["D"] # <-- IMPORTANT: Run one city at a time
COLUMNS = ["uid","d","t","x","y"]
DTYPES = {"uid":"int32","d":"int16","t":"int16","x":"int16","y":"int16"}

TRAIN_DAY_MAX = 60
TEST_DAY_MIN = 61
TEST_DAY_MAX = 75
MASK_VALUE = 999
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

TARGET_RANGES = {
    "A":(147001,150000), "B":(27001,30000), "C":(22001,25000), "D":(17001,20000)
}

# --- Model Hyperparameters ---
TOP_K_CANDIDATES = 5      # The "committee" size.
VAL_USER_FRACTION = 0.5   # Hold out 50% of "future" users for validation.
VAL_CLUE_WINDOW = 10      # Artificially use t < 10 as the "clue" for validation.
POI_TOP_N_PER_HOUR = 20   # How many POIs to identify for each hour
POI_WEIGHT_BOOST = 1.5    # How much to boost POI matches in strategy_3
MAX_T = 23                # Max hour in a day (0-23)

# --- Output Configuration ---
OUT_DIR = "./ppm_ensemble_results_fixed"
os.makedirs(OUT_DIR, exist_ok=True)

MAKE_SUBMISSION = True # Set to True to generate the final submission file
RUN_FULL_FINAL = True  # Set to True to run the final training and prediction

# =============================================================================
# 1. HELPER FUNCTIONS
# =============================================================================

def load_city_df(city):
    """Loads the CSV data for a given city."""
    path = os.path.join(DATA_DIR, f"city_{city}_challengedata.csv")
    if not os.path.exists(path):
        path_lower = os.path.join(DATA_DIR, f"city_{city.lower()}_challengedata.csv")
        if not os.path.exists(path_lower):
            raise FileNotFoundError(f"Missing data: {path} or {path_lower}")
        path = path_lower
        
    print(f"Loading data from: {path}")
    return pd.read_csv(path, usecols=COLUMNS, dtype=DTYPES)

def to_loc_id(x, y, max_y):
    """Converts (x, y) to a single integer loc_id."""
    return int(x * (max_y + 1) + y)

def from_loc_id(loc_id, max_y):
    """Converts a single integer loc_id back to (x, y)."""
    if max_y <= 0: return (0, 0) # Edge case for empty or 1-cell data
    x = loc_id // (max_y + 1)
    y = loc_id % (max_y + 1)
    return int(x), int(y)

# =============================================================================
# 2. MODEL ARTIFACT BUILDERS
# =============================================================================

def build_ppm_profiles(history_df, max_y):
    """
    PHASE 1: Offline User Profiling.
    Builds profiles from all available training history.
    """
    print(f"Building {history_df['uid'].nunique()} user profiles...")
    user_profiles = {}
    if history_df.empty:
        return user_profiles

    user_groups = history_df.groupby('uid')

    for uid, user_df in tqdm(user_groups, desc="Profiling Users"):
        profile = { "fallback_loc": (0, 0), "day_signatures": defaultdict(dict) }
        if user_df.empty:
            user_profiles[int(uid)] = profile
            continue
        
        loc_id_counts = Counter(
            user_df.apply(lambda row: to_loc_id(row['x'], row['y'], max_y), axis=1)
        )
        
        if loc_id_counts:
            fallback_loc_id = loc_id_counts.most_common(1)[0][0]
            profile["fallback_loc"] = from_loc_id(fallback_loc_id, max_y)

        for _, row in user_df.iterrows():
            profile["day_signatures"][row['d']][row['t']] = to_loc_id(row['x'], row['y'], max_y)
        
        user_profiles[int(uid)] = profile
    
    print(f"Profiling complete. {len(user_profiles)} profiles built.")
    return user_profiles

def compute_timestamp_pois(history_df, max_y, top_n=20):
    """
    Computes the Top N POIs for each timestamp (hour) separately.
    """
    print(f"Computing Top {top_n} POIs per timestamp...")
    poi_sets = defaultdict(set)
    if history_df.empty:
        return poi_sets
        
    for t, group in tqdm(history_df.groupby('t'), desc="Computing POIs"):
        loc_ids = group.apply(lambda row: to_loc_id(row['x'], row['y'], max_y), axis=1)
        poi_counts = Counter(loc_ids)
        top_pois = poi_counts.most_common(top_n)
        poi_sets[t] = set([loc_id for loc_id, count in top_pois])
    print("POI computation complete.")
    return poi_sets

# =============================================================================
# 3. CORE PREDICTION STRATEGIES
# =============================================================================

def _get_top_k_days(profile, clue_profile, score_func):
    """Helper to score and sort past days based on a scoring function."""
    if not clue_profile:
        return [] # No clue, no matches
        
    historical_scores = defaultdict(float)
    for past_day, day_signature in profile["day_signatures"].items():
        score = score_func(day_signature, clue_profile)
        if score > 1e-9: # Only store non-zero scores
            historical_scores[past_day] = score
    
    best_past_days = sorted(
        historical_scores, key=historical_scores.get, reverse=True
    )
    return [profile["day_signatures"].get(d, {}) for d in best_past_days[:TOP_K_CANDIDATES]]

def _predict_with_voting(slots_to_predict, candidate_trajectories, clue_profile, fallback_loc_id, last_known_loc, last_known_time):
    """
    Helper to perform voting, including the advanced tie-breaker.
    """
    day_predictions = []
    
    for time in slots_to_predict:
        votes = [traj[time] for traj in candidate_trajectories if time in traj]
        
        if votes:
            vote_counts = Counter(votes)
            most_common = vote_counts.most_common()
            top_vote_count = most_common[0][1]
            tied_locs = set([loc for loc, count in most_common if count == top_vote_count])
            
            if len(tied_locs) == 1:
                final_loc_id = tied_locs.pop()
            else:
                # TIE-BREAKER LOGIC: Check for spatio-temporal continuity
                winner = None
                if last_known_loc != -1:
                    for traj in candidate_trajectories:
                        if traj.get(time) in tied_locs and traj.get(last_known_time) == last_known_loc:
                            winner = traj.get(time)
                            break # Found a day that supports this transition
                
                if winner is not None:
                    final_loc_id = winner
                else:
                    final_loc_id = list(tied_locs)[0] # Default to first
        else:
            final_loc_id = fallback_loc_id
        
        day_predictions.append((time, final_loc_id))
        
        # Update last known for the next iteration
        last_known_time = time
        last_known_loc = final_loc_id
        
    return day_predictions


# --- Strategy 1: Baseline PPM (Exact Match) ---
def strategy_1(uid, clue_profile, slots_to_predict, user_profiles, max_y, **kwargs):
    profile = user_profiles.get(uid, {"fallback_loc": (0, 0), "day_signatures": {}})
    fallback_loc_id = to_loc_id(profile["fallback_loc"][0], profile["fallback_loc"][1], max_y)

    def score_func(day_signature, clue):
        score = 0
        for time, loc_id in clue.items():
            if day_signature.get(time) == loc_id:
                score += 1
        return float(score)

    candidate_trajectories = _get_top_k_days(profile, clue_profile, score_func)
    
    last_known_time = max(clue_profile.keys(), default=-1)
    last_known_loc = clue_profile.get(last_known_time, -1)
    
    return _predict_with_voting(slots_to_predict, candidate_trajectories, clue_profile, fallback_loc_id, last_known_loc, last_known_time)


# --- Strategy 2: cosine_time ---
# Cosine Similarity on a vector of *matches per time slot*.
def strategy_2(uid, clue_profile, slots_to_predict, user_profiles, max_y, **kwargs):
    profile = user_profiles.get(uid, {"fallback_loc": (0, 0), "day_signatures": {}})
    fallback_loc_id = to_loc_id(profile["fallback_loc"][0], profile["fallback_loc"][1], max_y)
    
    clue_times = clue_profile.keys()
    clue_vec = np.zeros(MAX_T + 1)
    for t in clue_times:
        clue_vec[t] = 1.0  # Vector of all 1s for clue times
    
    if not clue_times: # Handle empty clue
        candidate_trajectories = []
    else:
        def score_func(day_signature, clue):
            past_day_vec = np.zeros(MAX_T + 1)
            for t in clue_times:
                # ONLY put a 1 if the location MATCHES
                if day_signature.get(t) == clue.get(t):
                    past_day_vec[t] = 1.0
            
            if np.sum(past_day_vec) == 0:
                return 0.0
            
            # 
            # This is now a true similarity of *matching time slots*
            return cosine_similarity([clue_vec], [past_day_vec])[0][0]

        candidate_trajectories = _get_top_k_days(profile, clue_profile, score_func)
    
    last_known_time = max(clue_profile.keys(), default=-1)
    last_known_loc = clue_profile.get(last_known_time, -1)
    
    return _predict_with_voting(slots_to_predict, candidate_trajectories, clue_profile, fallback_loc_id, last_known_loc, last_known_time)


# --- Strategy 3: cosine_poi_time ---
# Cosine Similarity on *matches per time slot*, with POI matches *boosted*.
def strategy_3(uid, clue_profile, slots_to_predict, user_profiles, max_y, poi_sets, **kwargs):
    profile = user_profiles.get(uid, {"fallback_loc": (0, 0), "day_signatures": {}})
    fallback_loc_id = to_loc_id(profile["fallback_loc"][0], profile["fallback_loc"][1], max_y)
    
    clue_times = clue_profile.keys()
    clue_vec = np.zeros(MAX_T + 1)
    for t in clue_times:
        clue_vec[t] = 1.0 # Vector of all 1s for clue times
    
    if not clue_times: # Handle empty clue
        candidate_trajectories = []
    else:
        def score_func(day_signature, clue):
            past_day_vec = np.zeros(MAX_T + 1)
            for t in clue_times:
                # Check for a match
                if day_signature.get(t) == clue.get(t):
                    # If it matches, check if it's a POI
                    if clue.get(t) in poi_sets.get(t, set()):
                        past_day_vec[t] = POI_WEIGHT_BOOST # Boosted weight for POI match
                    else:
                        past_day_vec[t] = 1.0 # Normal weight for non-POI match
            
            if np.sum(past_day_vec) == 0:
                return 0.0
            
            return cosine_similarity([clue_vec], [past_day_vec])[0][0]

        candidate_trajectories = _get_top_k_days(profile, clue_profile, score_func)
    
    last_known_time = max(clue_profile.keys(), default=-1)
    last_known_loc = clue_profile.get(last_known_time, -1)

    return _predict_with_voting(slots_to_predict, candidate_trajectories, clue_profile, fallback_loc_id, last_known_loc, last_known_time)

# =============================================================================
# 4. PIPELINE & EVALUATION FUNCTIONS
# =============================================================================

def run_model_evaluation(val_df, user_profiles, max_y, prediction_strategy, **strategy_kwargs):
    """
    Runs a simulation on a validation set for a given prediction_strategy.
    """
    predictions_list = []  # For GeoBLEU
    ground_truth_list = [] # For GeoBLEU
    
    grouped = val_df.groupby(['uid', 'd'])
    
    desc = f"Validating ({prediction_strategy.__name__})"
    for (uid, day), group in tqdm(grouped, desc=desc):
        uid = int(uid)
        clue_profile = {}
        targets_to_predict = [] # List of (time, true_x, true_y)
        
        for _, row in group.iterrows():
            loc_id = to_loc_id(row['x'], row['y'], max_y)
            if row['t'] < VAL_CLUE_WINDOW:
                clue_profile[row['t']] = loc_id
            else:
                targets_to_predict.append((row['t'], row['x'], row['y']))
        
        if not targets_to_predict:
            continue

        target_slots = [t for t, x, y in targets_to_predict]
        
        # Call the injected strategy function
        day_predictions_loc_id = prediction_strategy(
            uid, clue_profile, target_slots, user_profiles, max_y, **strategy_kwargs
        )
        
        day_predictions_xy = [
            (t, from_loc_id(loc_id, max_y)[0], from_loc_id(loc_id, max_y)[1]) 
            for t, loc_id in day_predictions_loc_id
        ]
        
        # Collate predictions and ground truth for GeoBLEU
        for (time_p, pred_x, pred_y), (time_g, true_x, true_y) in zip(day_predictions_xy, targets_to_predict):
            predictions_list.append((uid, day, time_p, pred_x, pred_y))
            ground_truth_list.append((uid, day, time_g, true_x, true_y))
            
    return predictions_list, ground_truth_list

def ensemble_predictions(pred_lists_map, ground_truth_list):
    """
    Combines multiple prediction lists using a *weighted* voting system.
    """
    print("\nEnsembling model predictions (Weighted)...")
    if not ground_truth_list or not pred_lists_map:
        return 0.0

    # --- NEW: Define weights for each model ---
    # Give more weight to the robust baseline (PPM) and the
    # new fixed cosine model (strategy_2).
    model_weights = {
        "strategy_1": 2,  # PPM baseline
        "strategy_2": 2,  # Fixed cosine_time
        "strategy_3": 1,  # Fixed cosine_poi_time
    }
    # 
    # ----------------------------------------

    final_votes = defaultdict(Counter)
    key_to_gt = {}
    
    # Create a map of ground truth for easy lookup
    for uid, d, t, x, y in ground_truth_list:
        key_to_gt[(uid, d, t)] = (x, y)
    
    # Cast weighted votes from each model
    for model_name, pred_list in pred_lists_map.items():
        weight = model_weights.get(model_name, 1) # Get the model's weight
        for uid, d, t, x, y in pred_list:
            key = (uid, d, t)
            if key in key_to_gt: # Only vote on points we can score
                final_votes[key][(x, y)] += weight # Add the weighted vote
                
    final_pred_list = []
    final_gt_list = []
    
    # Tally votes
    for key, votes in final_votes.items():
        (top_x, top_y), _ = votes.most_common(1)[0]
        
        final_pred_list.append((key[0], key[1], key[2], top_x, top_y))
        gt_x, gt_y = key_to_gt[key]
        final_gt_list.append((key[0], key[1], key[2], gt_x, gt_y))
        
    if not final_pred_list:
        print("Ensembling resulted in no predictions.")
        return 0.0

    # Score the ensembled predictions
    score = calc_geobleu_bulk(final_pred_list, final_gt_list, processes=1)
    return score

# =============================================================================
# 5. MAIN EXECUTION (VALIDATION & ENSEMBLING)
# =============================================================================

def main(smoke_test=False):
    total_start = time.time()
    
    for city in CITIES:
        print("\n\n" + "="*50)
        print("RUNNING CITY:", city)
        print("="*50 + "\n")
        
        # --- 1. Load Data ---
        df = load_city_df(city)
        
        if smoke_test:
            print(f"SMOKE TEST: Sampling down to 1% of data.")
            df = df.sample(frac=0.01, random_state=RANDOM_SEED)

        labeled_df = df[df['x'] != MASK_VALUE]
        if labeled_df.empty:
            print(f"No labeled data found for city {city}. Skipping.")
            continue
        MAX_Y = labeled_df['y'].max()
        del labeled_df
        gc.collect()
        
        # --- 2. User-Based Train/Val Split ---
        print("Performing user-based train/validation split...")
        future_labeled_df = df[(df['d'] >= TEST_DAY_MIN) & (df['x'] != MASK_VALUE)]
        future_labeled_users = future_labeled_df['uid'].unique()

        if len(future_labeled_users) == 0:
            print("No labeled users found in future data for validation. Skipping.")
            continue

        future_train_users, val_users = train_test_split(
            future_labeled_users, test_size=VAL_USER_FRACTION, random_state=RANDOM_SEED
        )
        future_train_user_set = set(future_train_users)
        val_user_set = set(val_users)
        
        # --- 3. Build Artifacts for Validation ---
        history_df_1_60 = df[(df['d'] <= TRAIN_DAY_MAX) & (df['x'] != MASK_VALUE)]
        history_df_61_75 = future_labeled_df[future_labeled_df['uid'].isin(future_train_user_set)]
        full_history_df = pd.concat([history_df_1_60, history_df_61_75])
        
        del history_df_1_60, history_df_61_75
        gc.collect()
        
        # Build Profiles and POIs
        val_profiles = build_ppm_profiles(full_history_df, MAX_Y)
        val_poi_sets = compute_timestamp_pois(full_history_df, MAX_Y, top_n=POI_TOP_N_PER_HOUR)
        
        val_data_labeled = future_labeled_df[future_labeled_df['uid'].isin(val_user_set)]
        
        del full_history_df, future_labeled_df
        gc.collect()

        # --- 4. Run Individual Model Evaluations ---
        if val_data_labeled.empty or calc_geobleu_bulk is None:
            print("No validation data or GeoBLEU. Skipping all evaluations.")
            continue
            
        strategy_kwargs = {
            "user_profiles": val_profiles,
            "max_y": MAX_Y,
            "poi_sets": val_poi_sets
        }

        # Model 1: PPM
        pred_1, gt_1 = run_model_evaluation(val_data_labeled, prediction_strategy=strategy_1, **strategy_kwargs)
        score_1 = calc_geobleu_bulk(pred_1, gt_1, processes=1) if pred_1 else 0.0
        
        # Model 2: cosine_time (FIXED)
        pred_2, gt_2 = run_model_evaluation(val_data_labeled, prediction_strategy=strategy_2, **strategy_kwargs)
        score_2 = calc_geobleu_bulk(pred_2, gt_2, processes=1) if pred_2 else 0.0

        # Model 3: cosine_poi_time (FIXED)
        pred_3, gt_3 = run_model_evaluation(val_data_labeled, prediction_strategy=strategy_3, **strategy_kwargs)
        score_3 = calc_geobleu_bulk(pred_3, gt_3, processes=1) if pred_3 else 0.0

        # --- 5. Run Ensemble Evaluation ---
        pred_map = {"strategy_1": pred_1, "strategy_2": pred_2, "strategy_3": pred_3}
        # gt_1, gt_2, gt_3 are all identical, so we just pass one
        score_ensemble = ensemble_predictions(pred_map, gt_1) 

        # --- 6. Print All Results ---
        print("\n\n" + "="*50)
        print(f"CITY {city} - VALIDATION RESULTS (FIXED)")
        print("="*50)
        print(f"Model 1 (PPM)...............: GeoBLEU = {score_1:.6f}")
        print(f"Model 2 (cosine_time).......: GeoBLEU = {score_2:.6f}")
        print(f"Model 3 (cosine_poi_time)...: GeoBLEU = {score_3:.6f}")
        print("-"*50)
        print(f"FINAL ENSEMBLE (Weighted)....: GeoBLEU = {score_ensemble:.6f}")
        print("="*50)

        del val_profiles, val_poi_sets, val_data_labeled, pred_1, pred_2, pred_3, gt_1, gt_2, gt_3
        gc.collect()

        # --- 7. (Optional) Create Final Submission File ---
        if RUN_FULL_FINAL and MAKE_SUBMISSION and not smoke_test:
            print("\nCreating final submission file...")
            
            # Re-train on ALL labeled data
            all_labeled_data = df[df['x'] != MASK_VALUE]
            final_model_profiles = build_ppm_profiles(all_labeled_data, MAX_Y)
            final_poi_sets = compute_timestamp_pois(all_labeled_data, MAX_Y, top_n=POI_TOP_N_PER_HOUR)
            del all_labeled_data
            gc.collect()
            
            # Get all data for prediction (clues + masks)
            data_for_prediction = df[df['d'] >= TEST_DAY_MIN].copy()
            
            # --- Generate 3 sets of submission predictions ---
            sub_strategy_kwargs = {
                "user_profiles": final_model_profiles,
                "max_y": MAX_Y,
                "poi_sets": final_poi_sets
            }
            
            def generate_submission(prediction_strategy, **kwargs):
                print(f"Generating submission for {prediction_strategy.__name__}...")
                final_predictions = []
                grouped = data_for_prediction.groupby(['uid', 'd'])
                desc = f"Predicting ({prediction_strategy.__name__})"
                
                for (uid, day), group in tqdm(grouped, desc=desc):
                    uid = int(uid)
                    clue_profile = {}
                    slots_to_predict = []
                    
                    for _, row in group.iterrows():
                        if row['x'] != MASK_VALUE:
                            clue_profile[row['t']] = to_loc_id(row['x'], row['y'], MAX_Y)
                        else:
                            slots_to_predict.append(row['t'])
                    
                    if slots_to_predict:
                        day_preds = prediction_strategy(uid, clue_profile, slots_to_predict, **kwargs)
                        for t, loc_id in day_preds:
                            x, y = from_loc_id(loc_id, MAX_Y)
                            final_predictions.append((uid, day, t, x, y))
                return final_predictions

            preds_sub_1 = generate_submission(strategy_1, **sub_strategy_kwargs)
            preds_sub_2 = generate_submission(strategy_2, **sub_strategy_kwargs)
            preds_sub_3 = generate_submission(strategy_3, **sub_strategy_kwargs)

            # --- Ensemble the 3 submission lists (Weighted) ---
            print("Ensembling submission files (Weighted)...")
            final_sub_votes = defaultdict(Counter)
            
            model_weights = {"strategy_1": 2, "strategy_2": 2, "strategy_3": 1}
            
            for (uid, d, t, x, y) in preds_sub_1:
                final_sub_votes[(uid, d, t)][(x, y)] += model_weights["strategy_1"]
            for (uid, d, t, x, y) in preds_sub_2:
                final_sub_votes[(uid, d, t)][(x, y)] += model_weights["strategy_2"]
            for (uid, d, t, x, y) in preds_sub_3:
                final_sub_votes[(uid, d, t)][(x, y)] += model_weights["strategy_3"]
                
            final_sub_rows = []
            for (uid, d, t), votes in final_sub_votes.items():
                (top_x, top_y), _ = votes.most_common(1)[0]
                final_sub_rows.append({"uid": uid, "d": d, "t": t, "x": top_x, "y": top_y})
                
            submission_df = pd.DataFrame(final_sub_rows)
            
            # Filter to *only* the users we must predict for
            lo, hi = TARGET_RANGES[city]
            target_users = set(df[(df['uid'].between(lo, hi)) & (df['x'] == MASK_VALUE)]['uid'].unique())
            submission_df = submission_df[submission_df['uid'].isin(target_users)]
            
            output_filename = os.path.join(OUT_DIR, f"submission_ensemble_fixed_{city}.csv")
            submission_df.to_csv(output_filename, index=False)
            print(f"Final ENSEMBLE submission file saved to {output_filename}")

        del df
        gc.collect()

    print("\nTotal elapsed (s):", int(time.time() - total_start))


# =============================================================================
# 6. SMOKE TEST
# =============================================================================
# Set SMOKE_TEST = True to run a fast 1% sample of the data.
# Set SMOKE_TEST = False to run the full script.
# =============================================================================
SMOKE_TEST = False

if __name__ == "__main__":
    try:
        main(smoke_test=SMOKE_TEST)
    except Exception as e:
        print(f"\n\n--- SCRIPT FAILED ---")
        print(f"Error: {e}")
        import traceback
        traceback.print_exc()

Installing/checking requirements...
  Preparing metadata (setup.py) ... done
Note: you may need to restart the kernel to use updated packages.
GeoBLEU imported successfully.


RUNNING CITY: D

Loading data from: /kaggle/input/humob-data/15313913/city_D_challengedata.csv
Performing user-based train/validation split...
Building 20000 user profiles...


Profiling Users:   0%|          | 0/20000 [00:00<?, ?it/s]

Profiling complete. 20000 profiles built.
Computing Top 20 POIs per timestamp...


Computing POIs:   0%|          | 0/48 [00:00<?, ?it/s]

POI computation complete.


...

Generating submission for strategy_2...


Predicting (strategy_2):   0%|          | 0/284970 [00:00<?, ?it/s]

Generating submission for strategy_3...


Predicting (strategy_3):   0%|          | 0/284970 [00:00<?, ?it/s]

Ensembling submission files (Weighted)...
Final ENSEMBLE submission file saved to ./ppm_ensemble_results_fixed/submission_ensemble_fixed_D.csv

Total elapsed (s): 11451
